# 06 - VLM vs PatchCore: the scorecard (Week 3, part 3)

CPU only (Accelerator None). Inputs: **mvtec-eval-vlm** and **mvtec-patchcore** outputs.

How to read the metrics:
- **AUROC** (both models): can the score rank defective images above good ones? 0.5 = coin flip, 1.0 = perfect.
- **PatchCore F1-max**: F1 at the best threshold picked *on eval*, which is optimistic and is the standard way it's reported.
- **VLM F1**: F1 of the VLM's own yes/no answer, with no tuning.
- **Type / location accuracy**: VLM only; PatchCore can't name or place a defect in words.

In [ ]:
import pandas as pd
from pathlib import Path
find = lambda name: next(Path('/kaggle/input').rglob(name))
vlm = pd.read_csv(find('vlm_eval_preds.csv'))
pc = pd.read_csv(find('patchcore_eval_scores.csv'))
tr = pd.read_csv(find('vlm_train_check.csv'))
key = lambda p: '/'.join(str(p).split('/')[-4:])     # category/test/type/file, independent of mount path
vlm['k'], pc['k'] = vlm.path.map(key), pc.path.map(key)
assert set(vlm.k) == set(pc.k) and len(vlm) == len(pc) == 863, 'the two models were not scored on the same images'
print('same 863 eval images for both models ✓')

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

def auroc(d):
    y = (d.label == 'defect').astype(int)
    return np.nan if y.nunique() < 2 else roc_auc_score(y, d.score)

def best_f1(d):
    """F1 at the best threshold chosen ON eval: optimistic, the standard 'F1-max' in anomaly detection."""
    p, r, _ = precision_recall_curve((d.label == 'defect').astype(int), d.score)
    return float(np.nanmax(2 * p * r / np.clip(p + r, 1e-9, None)))

def vlm_f1(d):
    """F1 of the VLM's own yes/no answer: no threshold tuning at all."""
    y = (d.label == 'defect').astype(int)
    pred = d.pred_defective.astype(str).str.lower().eq('true').astype(int)
    return f1_score(y, pred, zero_division=0)

def type_acc(d):
    dd = d[d.label == 'defect']
    return (dd.pred_type == dd.defect_type).mean() if len(dd) else np.nan

def loc_acc(d):
    dd = d[d.label == 'defect']
    return (dd.pred_location == dd.true_location).mean() if len(dd) else np.nan

def table(vlm, pc, by):
    rows = []
    for key, g in vlm.groupby(by):
        h = pc[pc[by] == key]
        rows.append({by: key, 'n_eval': len(g), 'n_defect': int((g.label == 'defect').sum()),
                     'PatchCore AUROC': auroc(h), 'VLM AUROC': auroc(g),
                     'PatchCore F1-max': best_f1(h), 'VLM F1': vlm_f1(g),
                     'VLM type acc': type_acc(g), 'VLM location acc': loc_acc(g)})
    return pd.DataFrame(rows).set_index(by).round(3)

def macro_summary(vlm, pc):
    """Overall and per-kind rows = MEAN of the per-category metrics (the standard MVTec reporting).
    Pooling all images into one AUROC is wrong for PatchCore: its scores are distances on a
    different scale in every category, so they can't be ranked against each other."""
    cat = table(vlm, pc, 'category')
    cat['kind'] = vlm.groupby('category').kind.first()
    agg = {c: 'mean' for c in cat.columns if c not in ('kind', 'n_eval', 'n_defect')}
    agg.update(n_eval='sum', n_defect='sum')
    overall = cat.agg(agg).to_frame(f'all (mean of {len(cat)} categories)').T
    by_kind = cat.groupby('kind').agg(agg)
    by_kind.index = [f'{k} (mean of {n} categories)' for k, n in cat.kind.value_counts().sort_index().items()]
    cols = [c for c in cat.columns if c != 'kind']
    return pd.concat([overall, by_kind])[cols].round(3), cat.drop(columns='kind')

## Overall and by kind
Each row is the **mean of the per-category scores**, the standard way MVTec AD results are reported.

In [ ]:
summary, by_cat_all = macro_summary(vlm, pc)
summary

## By category (toothbrush and transistor are low-confidence: few eval images)

In [ ]:
by_cat = table(vlm, pc, 'category').sort_values('VLM AUROC')
by_cat

## Did the predictions from Week 1 hold? (screw and grid hardest)

In [ ]:
rank = by_cat['VLM AUROC'].rank()
for c in ['screw', 'grid']:
    print(f"{c}: VLM AUROC {by_cat.loc[c, 'VLM AUROC']}, rank {int(rank[c])} of {len(rank)} (1 = worst)")

## Memorisation check: training images vs eval

In [ ]:
for name, d in [('TRAIN (100)', tr), ('EVAL (863)', vlm)]:
    y = (d.label == 'defect').astype(int)
    det = d.pred_defective.astype(str).str.lower().eq('true').astype(int)
    print(f"{name:12s} defective acc {(det == y).mean():.1%} | type acc {type_acc(d):.1%} | AUROC {auroc(d):.3f}")

## Chart

In [ ]:
import matplotlib.pyplot as plt
ax = by_cat[['PatchCore AUROC', 'VLM AUROC']].plot.barh(figsize=(8, 7), color=['#888888', '#2f5bd3'])
ax.axvline(0.5, ls='--', c='k', lw=1); ax.set_xlim(0.3, 1.0)
ax.set_xlabel('image-level AUROC (0.5 = coin flip)'); ax.set_title('PatchCore vs fine-tuned Qwen2-VL-2B, 863 eval images')
plt.tight_layout(); plt.savefig('/kaggle/working/auroc_by_category.png', dpi=150); plt.show()

In [ ]:
pd.concat([summary, by_cat]).to_csv('/kaggle/working/scorecard.csv')
print('saved scorecard.csv and auroc_by_category.png')